In [ ]:
import pandas as pd
from google.colab import files
uploaded = files.upload()

Saving realty_data[1].csv to realty_data[1].csv


In [ ]:
data = pd.read_csv("realty_data[1].csv")

In [ ]:
data.head()

,product_name,period,price,postcode,address_name,lat,lon,object_type,total_square,rooms,floor,city,settlement,district,area,description,source
0,"3-комнатная, 137 м²",NaN,63000000,127473.0,"2-й Щемиловский переулок, 5а",55.778894,37.608844,Квартира,137.0,3.0,6.0,Москва,NaN,Тверской район,NaN,Просторная квартира свободной планировки с пан...,ЦИАН
1,"Студия, 16,7 м²",NaN,3250000,108815.0,"Харлампиева, 46",55.551025,37.313054,Квартира,16.7,NaN,1.0,Москва,NaN,Филимонковское поселение,NaN,ВНИМАНИЕ! ОЧЕНЬ ПРИВЛЕКАТЕЛЬНОЕ ПРЕ...,Домклик
2,"3-комнатная, 76 м²",NaN,16004680,NaN,"ЖК Прокшино, 8 к4",55.594802,37.431264,Квартира,76.0,3.0,6.0,Москва,NaN,Сосенское поселение,NaN,"Apт.1684018. 0,01% - гибкая ипотека! Воспользу...",Яндекс.Недвижимость
3,"1-комнатная, 24 м²",NaN,7841776,NaN,"ЖК Прокшино, 6 к2",55.594332,37.428099,Квартира,24.0,1.0,10.0,Москва,NaN,Сосенское поселение,NaN,Продается однокомнатная квартира № 381 в новос...,Новострой-М
4,"3-комнатная, 126 м²",NaN,120000000,121352.0,"Давыдковская, 18",55.721097,37.464342,Квартира,126.0,3.0,16.0,Москва,NaN,Фили-Давыдково район,NaN,Шикарное предложение!\nПродаётся трёхкомнатная...,Домклик


In [ ]:
!pip install streamlit scikit-learn pandas numpy pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 90.1 MB/s eta 0:00:00


In [ ]:
!pip install pyngrok --quiet

In [17]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib

# Загрузка данных
data = pd.read_csv("realty_data[1].csv")

target_col = "price"

numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
if target_col in numeric_cols:
    numeric_cols.remove(target_col)

X = data[numeric_cols]
y = data[target_col]

# Обработка пропусков
X_filled = X.fillna(X.median())
cols_to_drop = X_filled.columns[X_filled.isna().any()].tolist()
if cols_to_drop:
    print("Удаляем колонки с пропусками:", cols_to_drop)
    X_filled = X_filled.drop(columns=cols_to_drop)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_filled, y, test_size=0.2, random_state=42)

# Масштабирование
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Обучение
model = LinearRegression()
model.fit(X_train_scaled, y_train)

score = model.score(X_test_scaled, y_test)
print(f"R^2 на тестовой выборке: {score:.3f}")

# Сохранение
joblib.dump(model, "model.pkl")
joblib.dump(scaler, "scaler.pkl")

feature_info = []
for col in X_filled.columns:
    feature_info.append({
        "name": col,
        "min": float(X_filled[col].min()),
        "max": float(X_filled[col].max()),
        "default": float(X_filled[col].median())
    })
joblib.dump(feature_info, "feature_info.pkl")
print("Файлы модели сохранены: model.pkl, scaler.pkl, feature_info.pkl")


Удаляем колонки с пропусками: ['period']
R^2 на тестовой выборке: 0.750
Файлы модели сохранены: model.pkl, scaler.pkl, feature_info.pkl


In [18]:
%%writefile app.py
import streamlit as st
import joblib
import numpy as np

st.title("Прогноз стоимости недвижимости")
st.write("Ползунки настроены по реальным данным из твоего CSV.")

model = joblib.load("model.pkl")
scaler = joblib.load("scaler.pkl")
feature_info = joblib.load("feature_info.pkl")

input_values = []

for feat in feature_info:
    name = feat["name"]
    min_val = max(0, feat["min"])
    max_val = feat["max"]
    default_val = feat["default"]

    if max_val - min_val < 1e-6:
        min_val = default_val - 1
        max_val = default_val + 1

    value = st.slider(
        label=name,
        min_value=min_val,
        max_value=max_val,
        value=default_val,
        step=(max_val - min_val) / 100,
        key=name
    )
    input_values.append(value)

input_array = np.array(input_values).reshape(1, -1)

if st.button("Предсказать стоимость"):
    input_scaled = scaler.transform(input_array)
    prediction = model.predict(input_scaled)[0]
    st.subheader("Результат прогноза")
    st.metric(label="Прогнозная стоимость", value=f"{prediction:,.0f}")
    st.write("Единицы измерения: как в твоём CSV")


Overwriting app.py


In [ ]:
!streamlit run app.py --server.address=0.0.0.0 --server.port=8502




2026-09-29 14:48:46.980 Uvicorn server started on 0.0.0.0:8502

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.28.0.12:8502
  External URL: http://8.229.49.39:8502



In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(8502)